# MTTR & SLA

**How long does a vulnerability actually live once you stop excluding what is still
open — and where is it slow?**

Two clocks run here. The headline one starts at **first detection** — *time since we
saw it*, the harsher and simpler reading. Beside it the `actionable` columns start when a
**vendor fix became available**, which is the clock a team can actually be held to; the gap
between the two is exposure nobody could have closed. Every row in this register was
ingested under `hasFix: true`, so where no fix date was recorded the actionable clock falls
back to first detection — a construction rather than a guess.

## How to read this notebook

Every cell answers one question and shows one thing. Run them in order the first time; after
that any cell can be re-run on its own.

**Set the widgets at the top before you run anything.** `catalog` has no default on purpose.
Set the notebook to **Run accessed commands** (the dropdown beside *Run all*) if you want a
widget change to re-run the cells that depend on it — otherwise you will change the filter and
read a chart drawn under the old one.

Everything here reads one scan, pinned in cell 1. Charts that span scans say so in their title.

In [ ]:
import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "panels.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import panels, figures, tiles

PAGE = {
    "group_by": ("subscription_name", panels.GROUP_DIMENSIONS),
    "lens": ("contribution", ["contribution", "median"]),
}

panels.declare_widgets(**PAGE)
ctx = panels.context(spark, **{name: str(spec[0]) for name, spec in PAGE.items()})
displayHTML(tiles.scan_zone_from(panels.last_scan(spark, ctx).first()))

## The headline, and what qualifies it

In [ ]:
displayHTML(
    tiles.mttr_hero(
        panels.mttr_headline(spark, ctx).first(),
        panels.sla_extras(spark, ctx).agg(
            {"open_past_sla": "sum", "mttr_p90": "max"}
        ).toDF("open_past_sla", "mttr_p90").first(),
    )
)

## Trends

One point per saved scan, on an axis that is linear in elapsed time — so a gap between
runs looks like a gap. **A break in a line is a scan with no value, never a zero.**

These four charts read across every scan in the register, not just the pinned one.

In [ ]:
figures.render(
    figures.describe(
        figures.trend(
            panels.trend(spark, ctx, ["km_median", "mttr_median"]).toPandas(),
            "scan_ts",
            [
                figures.Series("km_median", "Median (KM, all)", figures.ACCENT,
                               symbol="circle", fill=True),
                figures.Series("mttr_median", "Median (naive, closed)", "#64748b",
                               dash="6,4", symbol="square"),
            ],
            y_unit="days",
        ),
        "Median time to remediate per scan. The censoring-aware estimate runs above the "
        "naive one whenever findings are still open; where they meet, almost everything has closed.",
    )
)

In [ ]:
figures.render(
    figures.describe(
        figures.trend(
            panels.trend(spark, ctx, ["open", "resolved"]).toPandas(),
            "scan_ts",
            [
                figures.Series("open", "Open", "#b91c1c", symbol="circle"),
                figures.Series("resolved", "Resolved", "#15803d", dash="6,4",
                               symbol="square"),
            ],
        ),
        "Open and resolved findings per scan. The two series differ by colour, dash pattern and marker, so neither depends on telling red from green.",
    )
)

The next chart counts the findings the **API returned** in each scan, while the *Open
past SLA* tile at the top counts **ledger lifecycles** — which include everything that
has since silently disappeared from the API. The two will not agree, and the size of the
gap is the size of what a snapshot-only pipeline was missing. Findings with no SLA target
(`UNKNOWN`) leave both sides of the ratio rather than counting as comfortably in-SLA.

In [ ]:
figures.render(
    figures.describe(
        figures.trend(
            panels.open_past_sla_trend(spark, ctx).toPandas(),
            "scan_ts",
            [figures.Series("open_past_sla", "Open past SLA", "#b91c1c")],
        ),
        "Findings still open past their severity's SLA target, per scan, counted over that scan's API snapshot.",
    )
)

## Distribution

S(t) is the share of the register still open at each elapsed time. It is drawn as a
**staircase** because survival is constant between events — a straight interpolation
would show findings closing on days nothing happened.

Each marker is its own legend entry with its own shape. A marker whose value was never
reached — the usual fate of a censored median — is left off rather than drawn at zero.

In [ ]:
_curve, _markers = panels.km_curve_points(spark, ctx)
figures.render(
    figures.describe(
        figures.survival(
            _curve.toPandas(),
            [
                dict(m, **{k: v for k, v in zip(
                    ("label", "color", "symbol"),
                    (km["label"], km["color"], km["symbol"]),
                )})
                for m in _markers
                for km in figures.KM_MARKERS
                if km["key"] == m["key"]
            ],
        ),
        "Share of tracked lifecycles still open at each elapsed time, with the medians and the restricted mean marked. Computed over the severities selected above, so it can legitimately differ from the published OVERALL headline.",
    )
)

Chart ▸ Bar (stacked) · X=bucket · Y=findings · Group by=severity · Order=bucket_rank ·
Legend=bottom · X title=Time to resolve · Y title=Lifecycles

The same distribution as counts rather than as a curve. Buckets are inclusive on the
left: exactly one day is `<=1d`, 7.01 days is `8-30d`.

In [ ]:
display(panels.time_to_resolve_buckets(spark, ctx))

## Remediation by severity

`sla_pct` is empty rather than zero for a severity with no target — `UNKNOWN` has none,
and a confident `0.0%` there would be a number nobody computed.

`resolved_api` and `resolved_disappeared` split how each closure was *learned*. A register
whose closures are overwhelmingly inferred is telling you something about the data source
as much as about the security programme.

In [ ]:
%sql
SELECT severity, km_median, km_median_lower_bound, mttr_median,
       sla_target, sla_pct, open, open_age_p50, open_age_p90,
       resolved, resolved_api, resolved_disappeared
FROM   v_mttr
WHERE  severity <> 'OVERALL'
ORDER  BY sev_rank

### The same severities on the actionable clock

`mttr_actionable_median` measures from the moment a fix existed rather than from detection,
so it is the figure a team can defend; `mttr_median` above is the exposure figure, and it is
never the smaller of the two. `actionable_resolved` is the population that clock could price at all
— it is at most `resolved`, and the difference is what had no fix clock to measure from.

`awaiting_vendor_fix_count` counts open findings with no fix available. It reads **0** here,
and that is a measurement rather than a gap: the scan filter pins `hasFix: true`, so every
row had a fix at the moment it was ingested. Drop that filter and this column starts
reporting.

In [ ]:
%sql
SELECT severity, mttr_median, mttr_actionable_median, mttr_actionable_mean,
       actionable_resolved, actionable_age_p50, actionable_age_p90,
       sla_pct, actionable_sla_pct, awaiting_vendor_fix_count
FROM   v_mttr
WHERE  severity <> 'OVERALL'
ORDER  BY sev_rank

## By group

Two ways of asking the same question, chosen with the `lens` widget.

**Contribution** measures excess *finding-days* — how many days of exposure each group
costs relative to the register's own median. A group can be slow and tiny, or average and
enormous; this is the axis that says which one is dragging the headline. Its rule at zero
is drawn **solid**, because zero here is an origin, not a target anyone is failing.

**Median** ranks groups slowest-first against a dashed rule at the overall median, which
*is* a comparison.

In [ ]:
_order = panels.group_palette(spark, ctx, ctx.param('group_by'), top_n=8)
if ctx.param('lens') == 'median':
    _rows = panels.km_by(spark, ctx, ctx.param('group_by'), top_n=8).toPandas()
    _fig = figures.bars_with_reference(
        _rows,
        value="km_median",
        label=ctx.param('group_by'),
        order=_order,
        reference_style="rule",
        overall=_rows["overall_km_median"].iloc[0] if len(_rows) else None,
        x_title="median time to remediate",
    )
    _text = 'Median time to remediate per group, slowest first, against the register overall.'
else:
    _rows = panels.mttr_contribution(spark, ctx, ctx.param('group_by')).toPandas()
    _fig = figures.diverging_bars(
        _rows, value="excess_finding_days", label=ctx.param("group_by"), order=_order
    )
    _text = 'Excess finding-days each group costs relative to the overall median. Right of the rule is slower than the register, left is faster.'
figures.render(figures.describe(_fig, _text))

In [ ]:
display(
    panels.km_by(spark, ctx, ctx.param('group_by')).join(
        panels.sla_extras(spark, ctx, ctx.param('group_by')), ctx.param('group_by'), 'left'
    )
)